In [ ]:
from transformers import AutoImageProcessor, AutoModel


def load_processor_and_model(pretrained_model_name):
    processor = AutoImageProcessor.from_pretrained(pretrained_model_name)
    model = AutoModel.from_pretrained(
        pretrained_model_name,
        device_map="cuda:0",
    )

    return processor, model


DEFAULT_PRETRAINED_MODEL_NAME = "facebook/dinov3-vith16plus-pretrain-lvd1689m"

processor, model = load_processor_and_model(DEFAULT_PRETRAINED_MODEL_NAME)
model = model.to("cuda:0")

In [ ]:
import torch

def compute_cls_token(processor, model, image):
    # Pre-process inputs with the AutoImageProcessor pipeline
    inputs = processor(images=image, return_tensors="pt").to(model.device)

    # Get patch constants
    patch_size = model.config.patch_size
    batch_size, _, img_height, img_width = inputs.pixel_values.shape
    num_patches_height, num_patches_width = (
        img_height // patch_size,
        img_width // patch_size,
    )

    # Run inference
    with torch.inference_mode():
        outputs = model(**inputs)

    last_hidden_states = outputs.last_hidden_state
    cls_token = last_hidden_states[:, 0, :]

    return cls_token

In [ ]:
from torch import nn

feature_dim = 1280
num_classes = 11
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
head_model = nn.Sequential(nn.Linear(feature_dim, num_classes))

state_dict = torch.load('/home/kiran/BuiltMfg/Artifacts/linear_probe_dinov3.pth')
head_model.load_state_dict(state_dict)

head_model = head_model.to("cuda:0")

In [ ]:
from transformers.image_utils import load_image

EXAMPLE_FILE = '/home/kiran/BuiltMfg/Datasets/StitchingNet/stitchingnet-dataset/versions/6/A. Cotton-Poly/0. Normal/A_00_001.jpg'
image = load_image(EXAMPLE_FILE)

image

In [ ]:
with torch.inference_mode():
    cls_token = compute_cls_token(processor, model, image)
    logits = head_model(cls_token)

In [ ]:
logits, torch.softmax(logits, dim=1).tolist()

In [ ]:
head_model[0].weight[10,10]